# Phase 5 v7 Closure Attack Notebook

No file IO. One claim per cell. PASS/FAIL outputs and inline figures.

In [ ]:
import math, cmath
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sympy as sp

def K(N, sign=-1):
    r=np.arange(N).reshape((N,1)); s=np.arange(N).reshape((1,N))
    return (1/np.sqrt(N))*np.exp(sign*2j*np.pi*r*s/N)
def G(N):
    r=np.arange(N); return np.exp(1j*np.pi*r*r/N)
N=12
M=np.abs(K(N))
print("PASS: definitions created", M[0,0])
plt.imshow(M); plt.title("|K_12| constant transfer magnitude"); plt.colorbar(); plt.show()

In [ ]:
def R(N):
    M=np.zeros((N,N), complex)
    for i in range(N): M[i,(-i)%N]=1
    return M
def maxabs(A): return float(np.max(np.abs(A)))
def pol(N):
    r=np.arange(N).reshape((N,1)); s=np.arange(N).reshape((1,N)); g=G(N)
    return maxabs(g[(r+s)%N]/(g[r]*g[s])-np.exp(2j*np.pi*r*s/N))
rows=[]
for N in range(2,62,2):
    k=K(N); I=np.eye(N,dtype=complex); k2=k@k
    rows.append({'N':N,'unitarity':maxabs(k@k.conj().T-I),'reversal':maxabs(k2-R(N)),'fourth':maxabs(k2@k2-I),'polarization':pol(N)})
df=pd.DataFrame(rows); mx=df[['unitarity','reversal','fourth','polarization']].max().max()
print('PASS' if mx<1e-10 else 'FAIL', mx)
df.plot(x='N', y=['unitarity','reversal','fourth','polarization']); plt.yscale('log'); plt.title('Even-N residuals'); plt.show(); df.head()

In [ ]:
controls=[]
for N in [8,12,20,26,60]:
    I=np.eye(N,dtype=complex); target=K(N)
    controls.append({'N':N,'control':'remove dual chart','residual':maxabs(I-target)})
    wrong=np.sqrt(N)*target
    controls.append({'N':N,'control':'wrong normalization','residual':maxabs(wrong@wrong.conj().T-I)})
    controls.append({'N':N,'control':'remove self twist','residual':float(max(abs(1-z) for z in G(N)))})
cf=pd.DataFrame(controls); print('PASS' if (cf.residual>0.1).all() else 'FAIL', cf.residual.min())
cf.plot(x='control', y='residual', kind='bar'); plt.title('Negative controls fail as expected'); plt.show(); cf

In [ ]:
odd=[]
for N in [3,5,7,9,11,13,15,21,25]:
    mx=0
    for r in range(N):
        q1=r*r/(2*N); q2=(r+N)*(r+N)/(2*N)
        mx=max(mx, abs(cmath.exp(2j*math.pi*q1)-cmath.exp(2j*math.pi*q2)))
    odd.append({'N':N,'representative_invariance_residual':mx})
of=pd.DataFrame(odd); print('PASS' if (of.representative_invariance_residual>1.9).all() else 'FAIL')
of.plot(x='N', y='representative_invariance_residual', kind='bar'); plt.title('Odd-N self-twist boundary'); plt.show(); of

In [ ]:
I=sp.I; rows=[]
for N in [2,4,6,8,12]:
    M=sp.Matrix([[sp.exp(-2*sp.pi*I*r*s/N)/sp.sqrt(N) for s in range(N)] for r in range(N)])
    U=(M*M.conjugate().T-sp.eye(N)).evalf(50)
    Rev=sp.Matrix([[1 if j==(-i)%N else 0 for j in range(N)] for i in range(N)])
    S=(M*M-Rev).evalf(50)
    rows.append({'N':N,'unitarity':max(abs(complex(U[i,j])) for i in range(N) for j in range(N)),'reversal':max(abs(complex(S[i,j])) for i in range(N) for j in range(N))})
sdf=pd.DataFrame(rows); print('PASS' if sdf[['unitarity','reversal']].max().max()<1e-40 else 'FAIL', sdf[['unitarity','reversal']].max().max()); sdf